# 01 — Document Ingestion



- Loads a PDF using PyMuPDF
- Splits text into overlapping chunks
- Converts chunks to embeddings using sentence-transformers
- Stores everything in ChromaDB (persisted to disk)

In [1]:
import fitz                          # PyMuPDF
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os

load_dotenv()   
print('All imports successful')

All imports successful


## Step 1 — Load the PDF
PyMuPDF opens the PDF page by page and extracts all text into one string.

In [2]:
def load_pdf(path):
    doc = fitz.open(path)
    full_text = ""
    for page_num, page in enumerate(doc):
        text = page.get_text()
        full_text += f"\n[Page {page_num + 1}]\n{text}"
    doc.close()
    return full_text

# ---- CHANGE THIS TO YOUR PDF PATH ----
PDF_PATH = "../docs/osnotes.pdf"

raw_text = load_pdf(PDF_PATH)

print(f"Total characters loaded : {len(raw_text):,}")
print(f"Estimated word count    : {len(raw_text.split()):,}")
print(f"\nFirst 500 characters:\n{'-'*40}")
print(raw_text[:500])

Total characters loaded : 107,414
Estimated word count    : 16,990

First 500 characters:
----------------------------------------

[Page 1]
24TU05MJC2 
Operating System 
L T  P  C 
Version 1.0 
 
2 1 0 3 
Pre-requisites/Exposure 
Data Structures Algorithm  
Co-requisites 
 
 
COURSE OBJECTIVES 
• To understand the fundamental concepts, architecture, and functions of modern 
operating systems. 
• To explore process management, CPU scheduling, synchronization, and deadlock 
handling mechanisms. 
• To study the principles of memory, file, and I/O management in operating systems. 
• To apply theoretical knowledge through simul


## Step 2 — Chunk the text
We split into 600-character chunks with 100-character overlap.

**Why overlap?** If a sentence is cut at a chunk boundary, it still appears
complete in the next chunk preserving context.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
    length_function=len
)

chunks = splitter.split_text(raw_text)

print(f"Total chunks created: {len(chunks)}")
print(f"Avg chunk length    : {sum(len(c) for c in chunks) // len(chunks)} chars")
print(f"\n--- Chunk 0 ---\n{chunks[0]}")
print(f"\n--- Chunk 1 (notice overlap with chunk 0) ---\n{chunks[1]}")

Total chunks created: 247
Avg chunk length    : 478 chars

--- Chunk 0 ---
[Page 1]
24TU05MJC2 
Operating System 
L T  P  C 
Version 1.0 
 
2 1 0 3 
Pre-requisites/Exposure 
Data Structures Algorithm  
Co-requisites 
 
 
COURSE OBJECTIVES 
• To understand the fundamental concepts, architecture, and functions of modern 
operating systems. 
• To explore process management, CPU scheduling, synchronization, and deadlock 
handling mechanisms. 
• To study the principles of memory, file, and I/O management in operating systems. 
• To apply theoretical knowledge through simulation, case studies, and practical 
problem-solving related to OS design. 
COURSE OUTCOMES (COS)

--- Chunk 1 (notice overlap with chunk 0) ---
problem-solving related to OS design. 
COURSE OUTCOMES (COS) 
After successful completion of the course, the students will be able to: 
• Explain the structure, functions, and components of operating systems.  
• Apply process scheduling and synchronization techniques to improve sy

## Step 3 — Create embeddings
Each chunk becomes a vector of 384 numbers that captures its meaning.
Chunks with similar meaning will have similar vectors.

In [4]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding chunks... (this takes 1-2 minutes on first run)")
embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"  → {embeddings.shape[0]} chunks, each with {embeddings.shape[1]} dimensions")
print(f"\nSample — first 8 numbers of chunk 0:")
print(embeddings[0][:8].round(4))

Embedding chunks... (this takes 1-2 minutes on first run)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Embeddings shape: (247, 384)
  → 247 chunks, each with 384 dimensions

Sample — first 8 numbers of chunk 0:
[-0.0302  0.0247 -0.0252 -0.033   0.0317 -0.1168  0.0222  0.0666]


## Step 4 — Store in ChromaDB
`PersistentClient` saves to disk so you never re-embed again.

In [5]:
client = chromadb.PersistentClient(path="../chroma_db")

# Clear old collection if re-running this notebook
try:
    client.delete_collection("my_docs")
    print("Cleared old collection")
except:
    pass

collection = client.create_collection(
    name="my_docs",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text
)

# Add in batches to avoid memory issues with large PDFs
BATCH_SIZE = 50
for i in range(0, len(chunks), BATCH_SIZE):
    batch_chunks = chunks[i:i+BATCH_SIZE]
    batch_embeds = embeddings[i:i+BATCH_SIZE]
    batch_ids    = [f"chunk_{j}" for j in range(i, i+len(batch_chunks))]
    collection.add(
        documents=batch_chunks,
        embeddings=batch_embeds.tolist(),
        ids=batch_ids
    )

print(f"Stored {collection.count()} chunks in ChromaDB")
print(f"Database saved to: ../chroma_db/")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Cleared old collection


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored 247 chunks in ChromaDB
Database saved to: ../chroma_db/


## Step 5 — Test retrieval
Verify the right chunks come back for a query. Lower distance = more relevant.

In [6]:
def test_retrieval(query, top_k=10):
    query_vec = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k
    )
    print(f"Query: {query}")
    print("-" * 50)
    for i, (doc, cid, dist) in enumerate(
        zip(results['documents'][0], results['ids'][0], results['distances'][0])
    ):
        print(f"\nRank {i+1} | {cid} | distance: {dist:.4f}")
        print(doc[:300])

# Change this query to something from YOUR document
test_retrieval("What are the generations of operating systems?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What are the generations of operating systems?
--------------------------------------------------

Rank 1 | chunk_12 | distance: 0.2779
[Page 5]
3. Ability to Evolve: An OS should be constructed in such a way as to permit 
the effective development, testing and introduction of new system functions 
at the same time without interfering with service. 
 
Operating system as User Interface – 
Every general-purpose computer consists of t

Rank 2 | chunk_8 | distance: 0.3007
[Page 3]
UNIT - I Introduction 
Concept  of  Operating   Systems- Generations   of Operating  
systems-Types of Operating Systems-OS  Services-System  
Calls-Structure  of  an  OS  -  Layered, Monolithic, 
Microkernel 
Operating 
Systems-Concept 
of 
Virtual 
Machine-Case study on UNIX and WINDOWS O

Rank 3 | chunk_2 | distance: 0.3536
part of a team-based project and communicate technical findings clearly. 
 
UNIT I – Introduction to Operating Systems                                                               

## Key observations
- Distance < 0.4 = strong match
- Distance > 1.0 = weak match — try rephrasing your query
- ChromaDB saved to `../chroma_db/` — do not delete this folder

**Next:** `02_vector_store_retrieval.ipynb` 